In [1]:
from pyspark.sql import functions as F
from infra.spark_utils import build_spark, normalize_ptbr_number
from infra.tickers_cache import get_tickers_price, update_tickers_cache

In [2]:
import sys
print(sys.executable)

c:\desenvolvimento\investment-visualizer\.venv\Scripts\python.exe


In [3]:
spark = build_spark()

In [4]:
import sys
import os

print("Notebook Python:", sys.executable)
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("PYSPARK_DRIVER_PYTHON:", os.environ.get("PYSPARK_DRIVER_PYTHON"))

Notebook Python: c:\desenvolvimento\investment-visualizer\.venv\Scripts\python.exe
PYSPARK_PYTHON: c:\desenvolvimento\investment-visualizer\.venv\Scripts\python.exe
PYSPARK_DRIVER_PYTHON: c:\desenvolvimento\investment-visualizer\.venv\Scripts\python.exe


In [5]:
spark = build_spark()
file_path = "../data/bronze/forms/economias.csv"
df = spark.read.csv(
    path = file_path,
    sep = ",", 
    header=True,
    multiLine=True # se n tiver quebra a logica 
)
df

DataFrame[timestamp: string, data_apuracao: string, instituicao_fin: string, resumo: string, aporte: string]

In [6]:
df = (df.withColumn(
    'resumo',
    F.regexp_replace(F.col('resumo'), r"\r\n|\r", "\n")
    )
    .withColumn(
    'resumo',
    F.split(F.col('resumo'), '\n')
    )
    .withColumn(
            'resumo',
            F.explode(F.col('resumo'))
        )
)
df.show()

+-------------------+-------------+---------------+--------------------+------+
|          timestamp|data_apuracao|instituicao_fin|              resumo|aporte|
+-------------------+-------------+---------------+--------------------+------+
|24/02/2026 10:21:10|   24/02/2026|             XP|Stock | BERK34 | ...|     0|
|24/02/2026 10:21:10|   24/02/2026|             XP|Renda Fixa | Teso...|     0|
|24/02/2026 10:22:11|   24/02/2026|          Clear|Stock | IVVB11 | ...|     0|
|24/02/2026 10:22:11|   24/02/2026|          Clear|Stock  | BOVA11 |...|     0|
|24/02/2026 10:22:11|   24/02/2026|          Clear|Stock | BERK34 | ...|     0|
|24/02/2026 10:23:26|   24/02/2026|         Nubank|Renda Fixa | Rese...|     0|
+-------------------+-------------+---------------+--------------------+------+



In [7]:
parts = F.split(F.col('resumo'), r"\|")
df = (df
      .withColumn("tipo", F.trim(parts.getItem(0)))
      .withColumn("nome", F.trim(parts.getItem(1)))
      .withColumn("qtd", F.trim(parts.getItem(2)))
      .withColumn("preco_medio", F.trim(parts.getItem(3)))
      .withColumn("preco_atual", F.trim(parts.getItem(4)))
      .withColumn("tipo", normalize_ptbr_number(F.col("tipo")))
      .withColumn("nome", normalize_ptbr_number(F.col("nome")))
      .withColumn("qtd", normalize_ptbr_number(F.col("qtd")))
      .withColumn("preco_medio", normalize_ptbr_number(F.col("preco_medio")))
      .withColumn("preco_atual", normalize_ptbr_number(F.col("preco_atual")))
    )

df.show()

+-------------------+-------------+---------------+--------------------+------+----------+--------------------+---+-----------+-----------+
|          timestamp|data_apuracao|instituicao_fin|              resumo|aporte|      tipo|                nome|qtd|preco_medio|preco_atual|
+-------------------+-------------+---------------+--------------------+------+----------+--------------------+---+-----------+-----------+
|24/02/2026 10:21:10|   24/02/2026|             XP|Stock | BERK34 | ...|     0|     Stock|              BERK34|  5|     120.02|       NULL|
|24/02/2026 10:21:10|   24/02/2026|             XP|Renda Fixa | Teso...|     0|Renda Fixa|             Tesouro|  1|   24000.00|   30000.00|
|24/02/2026 10:22:11|   24/02/2026|          Clear|Stock | IVVB11 | ...|     0|     Stock|              IVVB11| 15|     300.05|       NULL|
|24/02/2026 10:22:11|   24/02/2026|          Clear|Stock  | BOVA11 |...|     0|     Stock|              BOVA11| 10|      125.2|       NULL|
|24/02/2026 10:22:11

In [8]:
mapping_month = F.create_map(
    F.lit(1), F.lit("Jan"),  
    F.lit(2), F.lit("Fev"),  
    F.lit(3), F.lit("Mar"),  
    F.lit(4), F.lit("Abr"),  
    F.lit(5), F.lit("Mai"),  
    F.lit(6), F.lit("Jun"),  
    F.lit(7), F.lit("Jul"),  
    F.lit(8), F.lit("Ago"),  
    F.lit(9), F.lit("Set"),  
    F.lit(10), F.lit("Out"), 
    F.lit(11), F.lit("Nov"),
    F.lit(12), F.lit("Dec")
)

In [9]:
df = (df
      .withColumn(
          'data_apuracao',
          F.to_date(F.col('data_apuracao'), format='dd/MM/yyyy')
      )
      .withColumn(
          'ano',
          F.year(F.col('data_apuracao'))
      )
      .withColumn(
          'ano',
          F.year(F.col('data_apuracao'))
      )
      .withColumn(
          'mes_num',
          F.month(F.col('data_apuracao'))
      )
      .withColumn(
          'mes',
          mapping_month[F.col('mes_num')]
      )
)

df.show()


+-------------------+-------------+---------------+--------------------+------+----------+--------------------+---+-----------+-----------+----+-------+---+
|          timestamp|data_apuracao|instituicao_fin|              resumo|aporte|      tipo|                nome|qtd|preco_medio|preco_atual| ano|mes_num|mes|
+-------------------+-------------+---------------+--------------------+------+----------+--------------------+---+-----------+-----------+----+-------+---+
|24/02/2026 10:21:10|   2026-02-24|             XP|Stock | BERK34 | ...|     0|     Stock|              BERK34|  5|     120.02|       NULL|2026|      2|Fev|
|24/02/2026 10:21:10|   2026-02-24|             XP|Renda Fixa | Teso...|     0|Renda Fixa|             Tesouro|  1|   24000.00|   30000.00|2026|      2|Fev|
|24/02/2026 10:22:11|   2026-02-24|          Clear|Stock | IVVB11 | ...|     0|     Stock|              IVVB11| 15|     300.05|       NULL|2026|      2|Fev|
|24/02/2026 10:22:11|   2026-02-24|          Clear|Stock  

In [10]:
tickers_internacionais = ['BERK34', 'IVVB11', 'AAPL34', 'MSFT34', 'NVDC34', 'GOGL34', 'AMZO34', 'TSLA34', 'META34', 'MELI34']

df = (df
      .withColumn(
          'exposicao', 
          F.when(F.col('nome').isin(tickers_internacionais), "internacional")
          .otherwise("nacional")
      )
      .withColumn(
          'tipo',
          F.lower(F.col('tipo'))
      )
      .withColumn(
          'instituicao_fin',
          F.upper(F.col('instituicao_fin'))
      )
      .withColumn(
          'tipo',
          F.lower(F.col('tipo'))
      )
      .withColumn(
          'nome',
          F.when(F.col('tipo') == 'stock', F.upper(F.col('nome')))
          .otherwise(F.lower(F.col('nome')))
      )
)

df.show()

+-------------------+-------------+---------------+--------------------+------+----------+--------------------+---+-----------+-----------+----+-------+---+-------------+
|          timestamp|data_apuracao|instituicao_fin|              resumo|aporte|      tipo|                nome|qtd|preco_medio|preco_atual| ano|mes_num|mes|    exposicao|
+-------------------+-------------+---------------+--------------------+------+----------+--------------------+---+-----------+-----------+----+-------+---+-------------+
|24/02/2026 10:21:10|   2026-02-24|             XP|Stock | BERK34 | ...|     0|     stock|              BERK34|  5|     120.02|       NULL|2026|      2|Fev|internacional|
|24/02/2026 10:21:10|   2026-02-24|             XP|Renda Fixa | Teso...|     0|renda fixa|             tesouro|  1|   24000.00|   30000.00|2026|      2|Fev|     nacional|
|24/02/2026 10:22:11|   2026-02-24|          CLEAR|Stock | IVVB11 | ...|     0|     stock|              IVVB11| 15|     300.05|       NULL|2026| 

In [11]:
df = (df.select(
    F.to_timestamp(F.col('timestamp'), "dd/MM/yyyy HH:mm:ss").alias('timestamp'),
    F.to_date(F.col('data_apuracao'), "dd/MM/yyyy").alias('data_apuracao'),
    F.col('ano').cast('int').alias('ano'),
    F.col('mes_num').cast('int').alias('mes_num'),
    F.col('mes').cast('string').alias('mes'),
    F.col('instituicao_fin').cast('string').alias('instituicao_fin'),
    F.col('resumo').cast('string').alias('resumo'),
    F.col('tipo').cast('string').alias('tipo'),
    F.col('nome').cast('string').alias('nome'),
    F.col('qtd').cast('double').alias('qtd'),
    F.col('preco_medio').cast('double').alias('preco_medio'),
    F.col('preco_atual').cast('double').alias('preco_atual'),
    F.col('aporte').cast('double').alias('aporte'),
    F.col('exposicao').cast('string').alias('exposicao')
    )
)
df.show()

+-------------------+-------------+----+-------+---+---------------+--------------------+----------+--------------------+----+-----------+-----------+------+-------------+
|          timestamp|data_apuracao| ano|mes_num|mes|instituicao_fin|              resumo|      tipo|                nome| qtd|preco_medio|preco_atual|aporte|    exposicao|
+-------------------+-------------+----+-------+---+---------------+--------------------+----------+--------------------+----+-----------+-----------+------+-------------+
|2026-02-24 10:21:10|   2026-02-24|2026|      2|Fev|             XP|Stock | BERK34 | ...|     stock|              BERK34| 5.0|     120.02|       NULL|   0.0|internacional|
|2026-02-24 10:21:10|   2026-02-24|2026|      2|Fev|             XP|Renda Fixa | Teso...|renda fixa|             tesouro| 1.0|    24000.0|    30000.0|   0.0|     nacional|
|2026-02-24 10:22:11|   2026-02-24|2026|      2|Fev|          CLEAR|Stock | IVVB11 | ...|     stock|              IVVB11|15.0|     300.05|  

In [12]:
df_test = get_tickers_price(df)
df_test.show()

[*********************100%***********************]  3 of 3 completed


+------+----------+-----+--------------------+
|ticker|data_preco|close|        extracted_at|
+------+----------+-----+--------------------+
|BERK34|2026-03-07| NULL|2026-03-07 09:48:...|
|IVVB11|2026-03-07| NULL|2026-03-07 09:48:...|
|BOVA11|2026-03-07| NULL|2026-03-07 09:48:...|
+------+----------+-----+--------------------+



In [13]:
test_df = spark.createDataFrame(
    [("BERK34", 120.5), ("BOVA11", None)],
    ["ticker", "close"]
)

test_df.show()

+------+-----+
|ticker|close|
+------+-----+
|BERK34|120.5|
|BOVA11| NULL|
+------+-----+



In [14]:
spark.stop()